### 1. Install local Hugging Face dependencies for JupyterHub / Ubuntu
After installation, restart the notebook kernel once.

The model is selected directly in the script by changing `MODEL_NAME`.

The model will be downloaded from Hugging Face on the first run and then reused from the local Hugging Face cache.

### 2. Choose a Hugging Face model inside the notebook

The notebook uses these default models:

```python
MODEL_NAME = "Qwen/Qwen3.5-0.8B"
MODEL_NAME = "Qwen/Qwen3.5-4B-Base"
MODEL_NAME = "nvidia/NVIDIA-Nemotron-Nano-9B-v2"


In [ ]:
!pip install -U transformers accelerate  safetensors sentencepiece huggingface_hub torch==2.9.0 torchvision==0.24.0 torchaudio==2.9.0 --index-url https://download.pytorch.org/whl/cu130 #for CUDA version 13.0
!pip install flash-linear-attention causal_conv1d
#!pip install torch==2.9.0 torchvision==0.24.0 torchaudio==2.9.0 --index-url https://download.pytorch.org/whl/cu126 #for CUDA version 12.6
#

Looking in indexes: https://download.pytorch.org/whl/cu130, https://artifacts.dell.com/artifactory/api/pypi/ailfc-1003745-pypi-prd-local/simple, https://artifacts.dell.com/artifactory/api/pypi/aia-1001238-pypi-prd-local/simple, https://artifacts.dell.com/artifactory/api/pypi/aiops-1002685-pypi-prd-local/simple
Looking in indexes: https://artifacts.dell.com/artifactory/api/pypi/python/simple, https://artifacts.dell.com/artifactory/api/pypi/ailfc-1003745-pypi-prd-local/simple, https://artifacts.dell.com/artifactory/api/pypi/aia-1001238-pypi-prd-local/simple, https://artifacts.dell.com/artifactory/api/pypi/aiops-1002685-pypi-prd-local/simple
  Installing build dependencies ... -

### 02-ai-workflows

In [ ]:
import os, json, torch
from pathlib import Path

os.environ["TORCH_CUDNN_SDPA_ENABLED"] = "0"

from transformers import AutoTokenizer, AutoModelForCausalLM


MODEL_NAME = "nvidia/NVIDIA-Nemotron-Nano-9B-v2"

HF_TOKEN = None

HF_CACHE_DIR = "./hf_cache"

MAX_INPUT_TOKENS = 2048
MAX_NEW_TOKENS = 300
TEMPERATURE = 0.7
TOP_P = 0.9

# Ked by GPU stale padala, prepni na True
FORCE_CPU = False

cache_path = None
if HF_CACHE_DIR:
    cache_path = Path(HF_CACHE_DIR).expanduser().resolve()
    cache_path.mkdir(parents=True, exist_ok=True)

print(f"Loading local Hugging Face model: {MODEL_NAME}")

tokenizer_kwargs = {"trust_remote_code": True}

model_kwargs = {"trust_remote_code": True,"attn_implementation": "eager"}

if cache_path:
    tokenizer_kwargs["cache_dir"] = str(cache_path)
    model_kwargs["cache_dir"] = str(cache_path)

if HF_TOKEN:
    tokenizer_kwargs["token"] = HF_TOKEN
    model_kwargs["token"] = HF_TOKEN

use_cuda = torch.cuda.is_available() and not FORCE_CPU

if use_cuda:
    model_kwargs["dtype"] = torch.float16
    model_kwargs["device_map"] = "auto"
    print("CUDA detected. Running model on GPU with eager attention.")
else:
    model_kwargs["torch_dtype"] = torch.float32
    print("CUDA disabled or not detected. Running model on CPU.")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME,**tokenizer_kwargs)

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME,**model_kwargs)

if not use_cuda:
    model = model.to("cpu")

model.eval()

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Model loaded locally.")


def run_local_llm(prompt: str) -> str:
    messages = [{"role": "user","content": prompt.strip()}]

    if getattr(tokenizer, "chat_template", None):
        input_text = tokenizer.apply_chat_template(messages,tokenize=False,add_generation_prompt=True)
    else:
        input_text = prompt.strip()

    inputs = tokenizer(input_text,return_tensors="pt",truncation=True,max_length=MAX_INPUT_TOKENS)

    device = next(model.parameters()).device
    inputs = {key: value.to(device)for key, value in inputs.items()}

    generation_kwargs = {
                        "max_new_tokens": MAX_NEW_TOKENS,
                        "do_sample": TEMPERATURE > 0,
                        "pad_token_id": tokenizer.pad_token_id,
                        "eos_token_id": tokenizer.eos_token_id,
                        "use_cache": True
                        }

    if TEMPERATURE > 0:
        generation_kwargs["temperature"] = TEMPERATURE
        generation_kwargs["top_p"] = TOP_P

    with torch.inference_mode():
        output_ids = model.generate(**inputs,**generation_kwargs)

    generated_ids = output_ids[0][inputs["input_ids"].shape[-1]:]

    return tokenizer.decode(generated_ids,skip_special_tokens=True).strip()


def generate_x_post(topic: str) -> str:
    prompt = f"""
                You are an expert social media manager, and you excel at crafting viral and highly engaging posts for X formerly Twitter.

                Your task is to generate a post that is concise, impactful, and tailored to the topic provided by the user.
                Avoid using hashtags and lots of emojis. A few emojis are okay, but not too many.

                Keep the post short and focused. Structure it in a clean, readable way, using line breaks and empty lines to enhance readability.

                Topic:
                {topic}
                """

    return run_local_llm(prompt)


def main():
    #usr_input = input("What should the post be about? ")
    usr_input = str("new car model")
    x_post = generate_x_post(usr_input)

    print("\nGenerated X post:")
    print(x_post)


if __name__ == "__main__":
    main()

### 02-using-openai-sdk

In [ ]:
if "run_local_llm" not in globals():
    raise RuntimeError("Run the first Hugging Face setup cell before this workflow.")

def generate_x_post(topic: str) -> str:
    prompt = f"""
                You are an expert social media manager, and you excel at crafting viral and highly engaging posts for X (formerly Twitter).

                Your task is to generate a post that is concise, impactful, and tailored to the topic provided by the user.
                Avoid using hashtags and lots of emojis (a few emojis are okay, but not too many).

                Keep the post short and focused, structure it in a clean, readable way, using line breaks and empty lines to enhance readability.

                Here's the topic provided by the user for which you need to generate a post:
                <topic>
                {topic}
                </topic>
            """

    return run_local_llm(prompt)


def main():
    #usr_input = input("What should the post be about? ")
    usr_input = str("new car model")
    x_post = generate_x_post(usr_input)

    print("\nGenerated X post:")
    print(x_post)


if __name__ == "__main__":
    main()

### 03-few-shot-prompting

In [ ]:
if "run_local_llm" not in globals():
    raise RuntimeError("Run the first Hugging Face setup cell before this workflow.")

def generate_x_post(topic: str) -> str:
    with open("post-examples.json", "r") as f:
        examples = json.load(f)

    examples_str = ""
    for i, example in enumerate(examples, 1):
        examples_str += f"""
                            <example-{i}>
                                    <topic>
                                    {example['topic']}
                                    </topic>

                                    <generated-post>
                                    {example['post']}
                                    </generated-post>
                            </example-{i}>
                        """

    prompt = f"""
                You are an expert social media manager, and you excel at crafting viral and highly engaging posts for X (formerly Twitter).

                Your task is to generate a post that is concise, impactful, and tailored to the topic provided by the user.
                Avoid using hashtags and lots of emojis (a few emojis are okay, but not too many).

                Keep the post short and focused, structure it in a clean, readable way, using line breaks and empty lines to enhance readability.

                Here's the topic provided by the user for which you need to generate a post:
                <topic>
                {topic}
                </topic>

                Here are some examples of topics and generated posts:
                <examples>
                    {examples_str}
                </examples>

                Please use the tone, language, structure , and style of the examples provided above to generate a post that is engaging and relevant to the topic provided by the user.
                Don't use the content from the examples!
                """

    return run_local_llm(prompt)


def main():
    #usr_input = input("What should the post be about? ")
    usr_input = str("new car model")
    x_post = generate_x_post(usr_input)
    print("Generated X post")
    print(x_post)


if __name__ == "__main__":
    main()

### 04-multi-step-multi-model

In [ ]:
import json, requests

if "run_local_llm" not in globals():
    raise RuntimeError("Run the first Hugging Face setup cell before this workflow.")

def get_website_html(url: str) -> str:
    try:
        response = requests.get(url)
        response.raise_for_status()  # Raise an error for bad responses
        return response.text
    except requests.RequestException as e:
        print(f"Error fetching the URL {url}: {e}")
        return ""


def extract_core_website_content(html: str) -> str:
    prompt=f"""
                You are an expert web content extractor. Your task is to extract the core content from a given HTML page.
                The core content should be the main text, excluding navigation, footers, and other non-essential elements like scripts etc.

                Here is the HTML content:
                <html>
                {html}
                </html>

                Please extract the core content and return it as plain text.
            """

    return run_local_llm(prompt)


def summarize_content(content: str) -> str:

    prompt=f"""
                You are an expert summarizer. Your task is to summarize the provided content into a concise and clear summary.

                Here is the content to summarize:
                <content>
                {content}
                </content>

                Please provide a brief summary of the main points in the content. Prefer bullet points and avoid unncessary explanations.
            """

    return run_local_llm(prompt)


def generate_x_post(summary: str) -> str:
    with open("post-examples.json", "r") as f:
        examples = json.load(f)

    examples_str = ""
    for i, example in enumerate(examples, 1):
        examples_str += f"""
                            <example-{i}>
                                    <topic>
                                    {example['topic']}
                                    </topic>

                                    <generated-post>
                                    {example['post']}
                            </generated-post>
                            </example-{i}>
                        """

    prompt = f"""
                You are an expert social media manager, and you excel at crafting viral and highly engaging posts for X (formerly Twitter).

                Your task is to generate a post based on a short text summary.
                Your post must be concise and impactful.
                Avoid using hashtags and lots of emojis (a few emojis are okay, but not too many).

                Keep the post short and focused, structure it in a clean, readable way, using line breaks and empty lines to enhance readability.

                Here's the text summary which you should use to generate the post:
                <summary>
                {summary}
                </summary>

                Here are some examples of topics and generated posts:
                <examples>
                {examples_str}
                </examples>

                Please use the tone, language, structure , and style of the examples provided above to generate a post that is engaging and relevant to the topic provided by the user.
                Don't use the content from the examples!
            """

    return run_local_llm(prompt)


def main():
    website_url = input("Website URL: ")
    website_url = str("https://www.mazda.sk/akcna-ponuka/akcna-ponuka/mazda3/")
    print("Fetching website HTML...")
    try:
        html_content = get_website_html(website_url)
    except Exception as e:
        print(f"An error occurred while fetching the website: {e}")
        return

    if not html_content:
        print("Failed to fetch the website content. Exiting.")
        return

    print("---------")
    print("Extracting core content from the website...")
    core_content = extract_core_website_content(html_content)
    print("Extracted core content:")
    print(core_content)

    print("---------")
    print("Summarizing the core content...")
    summary = summarize_content(core_content)
    print("Generated summary:")
    print(summary)

    print("---------")
    print("Generating X post based on the summary...")
    x_post = generate_x_post(summary)
    print("Generated X post:")
    print(x_post)


if __name__ == "__main__":
    main()